In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
%cd /content/drive/MyDrive/LLM_Course/Testing/Transfer Learning CV

print(os.listdir())


/content/drive/MyDrive/LLM_Course/Testing/Transfer Learning CV
['Project_Image_Classifier_Project.ipynb', 'predict.py', 'Project_Image_Classifier_Project.html', 'label_map.json', 'oxford_flower_1.h5']


# Fine-Tuning a Language Model for Healthcare Question Answering

**A practical guide for students using Ollama and Hugging Face.**

This notebook provides a comprehensive, step-by-step tutorial on fine-tuning a smaller language model for a specific domain. We will focus on the healthcare sector and create a question-answering model capable of responding to medical queries.

## 1. Introduction to Fine-Tuning

### What is Fine-Tuning?

Large Language Models (LLMs) are pre-trained on vast amounts of general text data, giving them a broad understanding of language. However, for specialized tasks or domains, their performance can be significantly improved by **fine-tuning**. Fine-tuning is the process of taking a pre-trained model and continuing its training on a smaller, task-specific dataset. This adapts the model's knowledge and capabilities to the new domain, leading to more accurate and relevant outputs.

### Why is Fine-Tuning Important?

- **Domain Adaptation:** Fine-tuning allows a general-purpose model to learn the specific jargon, entities, and relationships of a particular field, such as medicine, law, or finance.
- **Improved Performance:** A fine-tuned model will outperform a general model on tasks within its specialized domain.
- **Cost and Time Efficiency:** Training a model from scratch is computationally expensive and time-consuming. Fine-tuning offers a more efficient way to achieve high performance on specific tasks.

### Parameter-Efficient Fine-Tuning (PEFT)

Fine-tuning a full LLM can still be resource-intensive. **Parameter-Efficient Fine-Tuning (PEFT)** methods address this by only updating a small subset of the model's parameters. This significantly reduces the computational and storage costs while achieving performance comparable to full fine-tuning. We will be using **Low-Rank Adaptation (LoRA)**, a popular PEFT technique, in this tutorial.

## 2. Understanding LoRA (Low-Rank Adaptation)

### How LoRA Works

LoRA works by adding small, trainable rank decomposition matrices to the existing model weights. Instead of updating all the weights in the model, LoRA only trains these small matrices, which are then added to the frozen pre-trained weights during inference. This is like adding a small, specialized 'add-on module' to a large existing machine, where only the add-on needs to be adjusted, not the entire machine.

**Key LoRA Parameters:**

- **r (rank):** This is the most crucial parameter, representing the dimension of the low-rank matrices. A lower `r` means fewer trainable parameters and faster training, but potentially less expressive power. A higher `r` allows for more fine-grained adaptation but requires more resources. Typical values range from 8 to 64, with common choices like 8, 16, or 32 offering a good balance.
- **lora_alpha:** This is a scaling factor for the LoRA weights. It essentially controls the 'strength' of the adapter. A common practice is to set `lora_alpha` to be `2 * r`, which helps maintain a reasonable learning rate for the LoRA weights regardless of `r`.
- **lora_dropout:** A dropout probability applied to the LoRA layers to prevent overfitting, similar to how dropout works in regular neural networks.
- **target_modules:** These specify *which* parts of the original LLM's architecture will have LoRA layers applied to them. Common choices include the query (`q_proj`), key (`k_proj`), and value (`v_proj`) projection matrices in the attention mechanism, as these are critical for how the model understands and generates text. Applying LoRA to these layers allows the model to adapt its attention patterns to the new domain.

**Benefits:**
- Only 0.1-1% of parameters need to be trained, drastically reducing the computational burden.
- Significantly reduced memory requirements, making fine-tuning possible on consumer GPUs.
- Faster training times, as fewer parameters are being updated.
- Small adapter files (often just a few MB), which are easy to share and deploy on top of a base model.

## 3. Setup and Installation

First, let's install the necessary Python libraries. We will need `transformers` for loading the model, `datasets` for loading our data, `peft` for the LoRA implementation, `trl` for the training loop, and `bitsandbytes` for quantization.

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.8 MB/s eta 0:00:00


## 4. Loading the Dataset

We will use the `MedQuad` dataset from Hugging Face, which contains medical question-answer pairs. This dataset has 16,407 examples covering various medical topics.

**Dataset Structure:**
- `qtype`: Type of question (symptoms, treatment, prevention, etc.)
- `Question`: The medical question
- `Answer`: The detailed answer

In [ ]:
from datasets import load_dataset

# Load the MedQuad dataset
dataset_name = "keivalya/MedQuad-MedicalQnADataset"
dataset = load_dataset(dataset_name, split="train")

print(f"Dataset size: {len(dataset)} examples")
print("\nFirst example:")
print(dataset[0])

README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

Dataset size: 16407 examples

First example:
{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


## 5. Data Preprocessing

To prepare the data for fine-tuning, we need to format it into a prompt template that the model can understand. A good prompt helps the model learn the desired input-output structure, guiding it to generate answers in a consistent and useful format. This is crucial because the model will learn to associate the '### Question:' and '### Answer:' tags with the respective parts of the input and output.

We will create a simple prompt that clearly separates the question from the answer, making it easy for the model to parse during training and for us to query during inference.

For this demonstration, we'll use a smaller subset of the data (the first 1000 examples) to speed up training. In a real-world scenario, you would typically use the entire dataset or a much larger portion to achieve better generalization and performance. Using a subset here allows for quicker experimentation and resource efficiency during this tutorial.

In [ ]:
def create_prompt_single(sample):
    """Format a sample into a training prompt."""
    prompt = f"""### Question:
{sample['Question']}

### Answer:
{sample['Answer']}"""
    return prompt

# Let's see what a formatted prompt looks like
print("Example formatted prompt:")
print(create_prompt_single(dataset[0]))
print("\n" + "="*50 + "\n")

# For this demo, we'll use a subset of the data (first 1000 examples)
# In practice, you would use the full dataset
train_dataset = dataset.select(range(min(1000, len(dataset))))
print(f"Training on {len(train_dataset)} examples")

Example formatted prompt:
### Question:
Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?

### Answer:
LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.


Training on 1000 examples


## 6. Model Loading and Configuration

Now, let's load our base model. We will use `TinyLlama`, a small but powerful model perfect for demonstrations on consumer hardware. We will also use 4-bit quantization to further reduce the memory footprint.


### Quantization

**4-bit quantization** is a powerful technique that dramatically reduces the memory footprint and computational cost of large language models. It works by converting the model's weights from higher precision formats (like 32-bit or 16-bit floating point numbers) down to 4-bit integers. This means each weight now occupies only one-fourth to one-eighth of the memory it did before.

**Why 4-bit quantization?**
- **Reduced Memory Usage:** The primary benefit is a significant reduction in VRAM (GPU memory) consumption, often by up to 75%. This allows us to run larger models on hardware with limited memory, such as consumer GPUs or Google Colab's free tier.
- **Faster Inference:** Smaller model sizes can sometimes lead to faster inference times, as less data needs to be moved between memory and processing units.
- **Minimal Impact on Quality:** Modern 4-bit quantization techniques, like `NF4` (NormalFloat 4-bit) used by `bitsandbytes`, are designed to minimize the loss in model quality. While there might be a slight degradation compared to full precision models, it's often negligible for many fine-tuning tasks, especially when combined with PEFT methods like LoRA.
- **Enabling PEFT:** Quantization is often paired with PEFT methods because it makes fine-tuning large models feasible by further reducing the resources required.

In [ ]:
def ask_question_untuned(question):
    """Ask the untuned base model a medical question."""
    prompt = f"""### Question:
{question}
### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the answer part
    answer = response.split("### Answer:")[-1].strip()
    return answer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# For demo inference (before fine-tuning)
demo_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(demo_model)

print("Base model loaded successfully!")

# Test untuned model quickly
# Test with a sample question using the untuned model
test_question = "What are the symptoms of diabetes?"
print(f"\nQuestion (Untuned Model): {test_question}")
print(f"\nAnswer (Untuned Model): {ask_question_untuned(test_question)}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Base model loaded successfully!

Question (Untuned Model): What are the symptoms of diabetes?

Answer (Untuned Model): 1. Blurred vision
2. Sores on the feet or hands
3. Swelling of the feet or ankles
4. Loss of feeling in the fingers or toes
5. Red, raw, or tender skin around the eyes
6. Itching or pain in the legs
7. Increased thirst or urination
8. Low blood sugar levels (hypoglycemia)
9. Fatigue
10. Weight loss
11. Changes in vision
12. Nausea or vomiting
13. Sweating at night
14. Fatigue or weakness
15. Rash or itching on the skin
16. Increased urine output
17. Poor wound healing
18. Sores or ulcers on the skin
19. Increased thirst and urination
20. Weakness and fatigue.

These are just some of the symptoms of diabetes. It's important to seek medical attention if you experience any of these symptoms or if you have concerns about your blood sugar levels.


In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # Changed from float16
    bnb_4bit_use_double_quant=True,
)

# Load the base model with quantization
training_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(training_model)

print("Base model loaded successfully!")

Base model loaded successfully!


## 7. Configuring LoRA

Now we'll configure the LoRA parameters and apply them to our model. This creates the trainable adapter layers that will be fine-tuned. The choices for these parameters are often based on a balance between performance, training speed, and available computational resources.

Here's a breakdown of the specific parameters used:

-   **`r=16` (rank):** We choose `r=16` as a moderately low rank. A lower rank means fewer parameters to train, leading to faster training and lower memory usage. While `r=8` might be even lighter, `r=16` often provides a good balance between parameter efficiency and the model's ability to learn the new domain-specific patterns.
-   **`lora_alpha=32`:** This is set to `2 * r`, which is a common heuristic. `lora_alpha` scales the LoRA weights, and setting it higher than `r` can sometimes help the LoRA layers have a stronger impact on the model's output, preventing the small LoRA updates from being "washed out" by the larger, frozen base model weights.
-   **`lora_dropout=0.05`:** A small dropout value is applied to the LoRA layers to introduce regularization and prevent overfitting, especially useful when fine-tuning on smaller datasets.
-   **`bias="none"`:** We set this to "none" meaning we do not train bias parameters with LoRA. For most applications, training only the weights (and not the biases) is sufficient for effective fine-tuning and further reduces the number of trainable parameters.
-   **`task_type="CAUSAL_LM"`:** This explicitly tells the `peft` library that we are performing Causal Language Modeling, which is the task of predicting the next token in a sequence. This is standard for models like TinyLlama used in generation tasks.
-   **`target_modules=["q_proj", "k_proj", "v_proj"]`:** These are the query, key, and value projection matrices within the attention heads of the transformer model. Applying LoRA to these modules is a common and effective strategy because they directly influence how the model interprets and relates different parts of the input sequence, making them crucial for adapting to new domain-specific language and relationships.

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,                      # Rank of the low-rank matrices
    lora_alpha=32,             # Scaling factor (usually 2x rank)
    lora_dropout=0.05,         # Dropout for regularization
    bias="none",               # Don't train bias parameters
    task_type="CAUSAL_LM",     # Task type: Causal Language Modeling
    target_modules=["q_proj", "k_proj", "v_proj"],  # Which layers to apply LoRA to
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 3,063,808 || all params: 1,103,112,192 || trainable%: 0.2777


Notice that only a small percentage of parameters are trainable! This is the power of PEFT - we're only training a tiny fraction of the model while still achieving good results.

## 8. Training the Model

We will use the `SFTTrainer` (Supervised Fine-Tuning Trainer) from the `trl` library to perform the fine-tuning. The `SFTTrainer` is specifically designed for supervised fine-tuning of large language models, making the training process simpler and more streamlined compared to using a generic `Trainer`. It handles common tasks like data formatting, tokenizer integration, and metric logging automatically, which is particularly helpful for domain adaptation tasks.

### Training Parameters:

-   **`output_dir`**: The directory where the model checkpoints and logs will be saved. `"./results"` is a common choice for local storage.
-   **`num_train_epochs=1`**: For this demonstration, we are setting the number of training epochs to 1. An epoch represents one full pass over the entire training dataset. While more epochs might lead to better performance, `1` is chosen here to keep the training time manageable for a tutorial, especially with a subset of the data. For real-world applications, you might experiment with 3-5 epochs or more.
-   **`per_device_train_batch_size=4`**: This determines the number of samples processed per GPU (or CPU) at once. A smaller batch size helps conserve memory, which is critical when working with large models and limited hardware. It means 4 examples are fed to the model in each training step on each device.
-   **`gradient_accumulation_steps=4`**: This technique effectively simulates a larger batch size without increasing memory usage. Instead of processing 4 examples and updating weights, we process 4 examples, accumulate gradients, and repeat this 4 times (4 * 4 = 16 effective batch size) before actually updating the model's weights. This allows us to use smaller `per_device_train_batch_size` while benefiting from the stability of a larger effective batch size.
-   **`learning_rate=2e-4`**: The learning rate controls how much the model's weights are adjusted with respect to the loss gradient. `2e-4` (0.0002) is a commonly used learning rate for fine-tuning LLMs with LoRA, as it's typically lower than rates used for full pre-training to avoid overshooting optimal weights.
-   **`bf16=True`**: This enables bfloat16 (Brain Floating Point) training. bfloat16 is a 16-bit floating-point format that offers a wider dynamic range than float16, making it more robust against overflow/underflow issues during training. It's especially useful when combined with 4-bit quantization, as the `bnb_4bit_compute_dtype` was set to `torch.bfloat16`, ensuring consistent computation precision.
-   **`logging_steps=10`**: Specifies how often (in steps) to log training metrics like loss to the console or tools like Weights & Biases (if integrated).
-   **`save_strategy="epoch"`**: The model will save a checkpoint at the end of each epoch.
-   **`optim="paged_adamw_8bit"`**: This specifies the optimizer to use. `paged_adamw_8bit` is a memory-efficient variant of the AdamW optimizer. It uses 8-bit optimizer states, further reducing memory consumption during training by offloading optimizer states to CPU memory and paging them to GPU as needed. This is crucial for training large models efficiently.
-   **`dataset_text_field="text"`**: This parameter is required by `SFTConfig` and tells the trainer which column in your dataset contains the text to be processed. In our `formatting_func`, we create a combined string which the `SFTTrainer` expects to be in a field called 'text' (this is handled internally by `formatting_func`).
-   **`max_length=512`**: This is the maximum sequence length that the tokenizer will process. Sequences longer than this will be truncated, and shorter ones will be padded. Choosing an appropriate `max_length` is important for managing memory and ensuring that relevant information is not cut off.
-   **`packing=False`**: When `False`, sequences are not packed together. Packing can improve training efficiency by concatenating multiple short examples into a single longer sequence to fully utilize the `max_length`. However, for instruction fine-tuning where clear separation of prompts and responses is critical, setting `packing=False` is often preferred to avoid unintended context leakage between examples.

**Train on completions only**


In [ ]:
from trl import SFTTrainer, SFTConfig


# Define training arguments
training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True, # using
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    dataset_text_field="text",  # Required for SFTConfig
    max_length=512,
    packing=False,  # Must be False for response template masking
)

# Format the dataset for training - handles SINGLE examples
def formatting_func(example):
    text = f"""### Question:
{example['Question']}

### Answer:
{example['Answer']}"""
    return text  # Return a single string, not a list

# Create the trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    formatting_func=formatting_func,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete!")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3136 > 2048). Running this sequence through the model will result in indexing errors


...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tatwan to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.845900
20,1.733100
30,1.668000
40,1.614700
50,1.591300
60,1.553900


Training complete!


## 9. Testing the Fine-Tuned Model

Let's test our fine-tuned model with a medical question to see how it performs!

In [ ]:
import torch

# After training, before inference:
# 1. Disable gradient checkpointing
model.gradient_checkpointing_disable()

# 2. Set to eval mode
model.eval()

# 3. Ensure model is in the correct dtype
model = model.to(dtype=torch.bfloat16)

# 4. Re-enable caching
model.config.use_cache = True

def ask_question(question):
    """Ask the fine-tuned model a medical question."""
    prompt = f"""### Question:
{question}

### Answer:
"""

    # Ensure inputs are in the same dtype as model
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("### Answer:")[-1].strip()
    return answer

# Test
test_question = "What are the symptoms of diabetes?"
print(f"Question: {test_question}")
print(f"\nAnswer: {ask_question(test_question)}")


Question: What are the symptoms of diabetes?

Answer: The most common symptom is weight loss. As the body loses its ability to use glucose, people with type 1 diabetes experience symptoms like hunger and fatigue. Some people also may have blurred vision, difficulty in walking or hearing, and pain in their feet.


## 10. Saving the Model

After training, we save the LoRA adapters. These are small files (typically just a few MB) that contain only the trained weights.

In [ ]:
# Save the LoRA adapters
output_dir = "tinyllama-medical-qa-lora"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")
print("\nYou can now load this model using:")
print("from peft import PeftModel")
print(f"model = PeftModel.from_pretrained(base_model, '{output_dir}')")

Model saved to tinyllama-medical-qa-lora

You can now load this model using:
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, 'tinyllama-medical-qa-lora')


## Loading the Model Later for use

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Configuration
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "tinyllama-medical-qa-lora"

# Configure 4-bit quantization (same as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model with quantization
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# Load LoRA adapters
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=False,
)

# Set for inference
model.eval()
model.config.use_cache = True

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

print("Model loaded successfully!")

# Test function
def ask_question(question):
    """Ask the fine-tuned model a medical question."""
    prompt = f"""### Question:
{question}

### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("### Answer:")[-1].strip()
    return answer

# Test it
print(ask_question("What are the symptoms of diabetes?"))


Model loaded successfully!
The symptoms of diabetes vary, depending on which type of diabetes a person has. 

In people with Type I (insulin-dependent) diabetes, symptoms include frequent urination and thirst, fatigue, hunger, weight loss, and blurred vision. In Type II (non-insulin-dependent) diabetes, symptoms may include feeling hungry or full, thirst, weight loss, tiredness, and swelling of feet, hands, legs, face, and neck.


## 11. Merging and Exporting for Ollama

To use the model with Ollama, we need to merge the LoRA adapters with the base model and convert it to GGUF format.

In [ ]:
from peft import PeftModel

# Load base model without quantization for merging
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Load and merge the LoRA weights
merged_model = PeftModel.from_pretrained(base_model, output_dir)
merged_model = merged_model.merge_and_unload()

# Save the merged model
merged_output_dir = "tinyllama-medical-qa-merged"
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)

print(f"Merged model saved to {merged_output_dir}")

Merged model saved to tinyllama-medical-qa-merged


## 12. Converting to GGUF and Using with Ollama

To use the model with Ollama, follow these steps in your terminal:

### Step 1: Install llama.cpp

```bash
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp
pip install -r requirements.txt
```

### Step 2: Convert to GGUF

```bash
python convert_hf_to_gguf.py /path/to/tinyllama-medical-qa-merged \
  --outfile tinyllama-medical-qa.gguf \
  --outtype f16
```

__Optional Quantization__

For smaller file size and faster inference, quantize to 4-bit or 8-bit:

```bash
# Build llama.cpp first
cd llama.cpp

# Install cmake if not available
apt-get update && apt-get install -y cmake

# Create build directory and compile
mkdir build
cd build
cmake ..
cmake --build . --config Release

# The quantization tool will be at:
# ./build/bin/llama-quantize

# Quantize to 4-bit (Q4_K_M is recommended)
./build/bin/llama-quantize tinyllama-medical-qa.gguf \
  tinyllama-medical-qa-q4.gguf Q4_K_M
```

### Step 3: Create a Modelfile

Create a file named `Modelfile` with this content:

```
FROM ./tinyllama-medical-qa-q4.gguf

TEMPLATE """### Question:
{{ .Prompt }}

### Answer:
"""

SYSTEM "You are a helpful medical assistant trained to answer healthcare questions."

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "### Question:"
```

### Step 4: Create and Run in Ollama
Install Ollama
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

Start the service

In [ ]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'




```bash
# Create the model in Ollama
ollama create tinyllama-medical-qa -f Modelfile

# Run the model
ollama run tinyllama-medical-qa
```

### Step 5: Test Your Model

```bash
ollama run tinyllama-medical-qa "What are the symptoms of diabetes?"
```

## 13. Summary and Key Takeaways

In this notebook, we successfully fine-tuned a small language model for healthcare question answering. Here are the key concepts we covered:

### What We Learned:

1. **Fine-Tuning Fundamentals:** Fine-tuning adapts a pre-trained model to a specific domain or task, improving performance without training from scratch.

2. **Parameter-Efficient Fine-Tuning (PEFT):** Methods like LoRA allow us to fine-tune models by training only a small fraction of parameters, making it feasible on consumer hardware.

3. **LoRA (Low-Rank Adaptation):** Adds small trainable matrices to frozen model weights, achieving comparable performance to full fine-tuning with minimal resources.

4. **Quantization:** 4-bit quantization reduces memory requirements by up to 75%, enabling larger models to run on limited hardware.

5. **Practical Workflow:**
   - Load dataset and preprocess with prompt templates
   - Load base model with quantization
   - Configure and apply LoRA
   - Train using SFTTrainer
   - Save, merge, and export for deployment

### Next Steps:

- **Experiment with different datasets:** Try fine-tuning on other domains like legal, financial, or technical documentation.
- **Adjust hyperparameters:** Experiment with different LoRA ranks, learning rates, and batch sizes.
- **Use larger models:** Apply the same techniques to larger models like Llama 2 7B or Mistral 7B.
- **Evaluate performance:** Create test sets and measure accuracy, relevance, and factual correctness.

### Resources:

- [Hugging Face PEFT Documentation](https://huggingface.co/docs/peft)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [Ollama Documentation](https://ollama.ai)
- [TRL Library](https://huggingface.co/docs/trl)